# Scratch: GSM8K and the `eval_min` helpers

Quick playground. Nothing here is part of the eval. Parsing and splits live in `finetuning.ipynb`; `eval_min` has no parser, only `to_rec` and the scorer.

In [1]:
# autoreload: edits to eval_min.py show up without restarting the kernel
%load_ext autoreload
%autoreload 2

import json, re, random
from collections import Counter

from datasets import load_dataset
import eval_min as em

/Users/pranjalbainsla/Desktop/ml/finetune/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
gsm = load_dataset("openai/gsm8k", "main")
train, test = gsm["train"], gsm["test"]
print(gsm)

Generating test split: 100%|██████████| 1319/1319 [00:00<00:00, 73502.16 examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 1319
    })
})


In [19]:
# look at one raw example
ex = train[:10]

for i in range(10):
    print(ex["question"][i], "\n---")
    print(ex["answer"][i], "\n...")


Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May? 
---
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72 
...
Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn? 
---
Weng earns 12/60 = $<<12/60=0.2>>0.2 per minute.
Working 50 minutes, she earned 0.2 x 50 = $<<0.2*50=10>>10.
#### 10 
...
Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided to give her $15 for that purpose, and her grandparents twice as much as her parents. How much more money does Betty need to buy the wallet? 
---
In the beginning, Betty has only 100 / 2 = $<<100/2=50>>50.
Betty's grandparents gave her 15 * 2 = $<<15*2=30>>30.
This means, Betty needs 100 - 50 - 30 - 15 = $<<100-50-30-15=5>>5 more.
#### 5 
...
J

In [ ]:
# a notebook-format parsed record (what finetuning.ipynb's parse() returns) -> what the scorer sees
p = {"question": train[3]["question"],
     "steps": [{"text": "x", "expr": "e", "val": v} for v in ("1,200", "48", "6")], "final": "6"}
rec = em.to_rec(p)
rec

In [ ]:
# raw train answers: how many have exactly one <<expr=val>> per line (rough look; the notebook's parse() is the real filter)
ok = [e for e in train if all(len(re.findall(r"<<.*?>>", l)) == 1 for l in e["answer"].rsplit("####", 1)[0].strip().split("\n"))]
print(f"one-annotation-per-line: {len(ok)} / {len(train)}")

In [ ]:
# a few examples that fail that shape, to see why the notebook's parse() rejects them
bad = [e for e in train if e not in ok]
for e in random.sample(bad, 3):
    print(e["answer"], "\n=====")

In [ ]:
# how many steps each level should reveal for this problem, and which numbers are in the question
r = rec
print("steps:", r["steps"], "answer:", r["answer"])
print("k per level:", {L: em.target_k(L, len(r["steps"])) for L in em.LEVELS})
print("numbers in question:", em.nums(r["question"]))

In [ ]:
# try the scorer on a hand-written reply
em.score("First, work out 48 / 2. What do you get?", r, 2)

## Dataset building

_(your code here)_